# Simple Semantic Kernel Agent Tutorial
# 简单的 Semantic Kernel 代理教程

Learn to build AI agents with Semantic Kernel in just a few steps. This tutorial covers the essentials: creating agents, adding tools, and managing conversations.
只需几个步骤，即可学习使用 Semantic Kernel 构建 AI 代理。本教程涵盖了基本要素：创建代理、添加工具和管理对话。

There are three foundational components in Semantic Kernel agents: 
Semantic Kernel 代理包含三个基本组件：
1. Agent Class: All agent types inherit from this class. Agent types include ChatCompletionAgent (uses standard chat completion APIs), OpenAIAssistantAgent (leverages OpenAI Assistant API with built in tools), AzureAIAgent (integrates with Azure AI services for enterprise scenarios), CopilotStudioAgent (connects to Microsoft Copilot Studio workflows). 
1. 代理类：所有代理类型都继承自此类。代理类型包括 ChatCompletionAgent（使用标准聊天完成 API）、OpenAIAssistantAgent（利用带有内置工具的 OpenAI Assistant API）、AzureAIAgent（针对企业场景与 Azure AI 服务集成）、CopilotStudioAgent（连接到 Microsoft Copilot Studio 工作流）。
2. Agent Thread: This handles how conversation history and state are maintained. This is important since agents need context from previous messages to make informed decisions. There are two approaches:
2. 代理线程：处理对话历史记录和状态的维护方式。这很重要，因为代理需要来自先前消息的上下文才能做出明智的决策。有两种方法：
    1. Service managed state: An agent service like Azure AI stores conversation history server-side, and accessed via thread ID.
    1. 服务管理状态：像 Azure AI 这样的代理服务在服务器端存储对话历史记录，并通过线程 ID 进行访问。
    2. Application managed state: Your application maintains the full chat history and passes it to the agent on each call. 
    2. 应用程序管理状态：你的应用程序维护完整的聊天记录，并在每次调用时将其传递给代理。
3. Agent orchestration: The framework provides pre-built patterns for coordinating multiple agents to handle complex workflows that single agents cannot manage effectively. 
3. 代理编排：该框架提供了预构建的模式，用于协调多个代理以处理单个代理无法有效管理的复杂工作流。
    1. Sequential: Agents execute one after the other in order. This is like document processing pipeline.
    1. 顺序：代理按顺序一个接一个地执行。这就像文档处理管道。
    2. Concurrent: Multiple agents working at the same time. Like customer inquiry handling (billing agent and account agent working in parallel).
    2. 并发：多个代理同时工作。就像处理客户咨询（计费代理和账户代理并行工作）。
    3. Handoff: Agents pass control to each other based on specialization. Like customer service triage to specialist agent.
    3. 移交：代理根据专业化相互传递控制权。就像客户服务分流给专家代理。
    4. Group chat: Agents collaborate in a shared conversation. Like project planning with domain experts. 
    4. 群聊：代理在共享对话中协作。就像与领域专家一起进行项目规划。

Agents leverage Semantic Kernel’s plugin system to access tools, databases, etc. 
代理利用 Semantic Kernel 的插件系统来访问工具、数据库等。

Agents can also be defined using YAML.
代理也可以使用 YAML 定义。

## Setup
## 设置

Install required packages and configure your environment:
安装所需的包并配置你的环境：

In [ ]:
# Install packages
!pip install semantic-kernel python-dotenv

# Import everything we need
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

print("✅ Setup complete!")

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Check if API key is configured
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("⚠️  Please add OPENAI_API_KEY to your .env file!")
    print("   Create a .env file with: OPENAI_API_KEY=your_actual_key_here")
else:
    print("✅ API key configured")

## Step 1: Create a Simple Agent
## 第一步：创建一个简单的代理

Let's start with the basics - an agent that can chat. We are simply giving the kernel (our orchestrator) access to chat service, which leverages the Chat Completion API endpoint from Open AI. We will add more tools later. 
让我们从基础开始——一个可以聊天的代理。我们只是让 Kernel（我们的编排器）访问聊天服务，该服务利用了 OpenAI 的聊天完成 API 端点。我们稍后会添加更多工具。

In [ ]:
# Create service and kernel
chat_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    api_key=OPENAI_API_KEY
)

kernel = Kernel()
kernel.add_service(chat_service)

# Create a basic agent
agent = ChatCompletionAgent(
    kernel=kernel,
    name="Assistant",
    instructions="You are a helpful and friendly assistant."
)

# Test it
response = await agent.get_response("Hello! What can you help me with?")
print(f"🤖 {agent.name}: {response.content}")

## Step 2: Add Tools (Functions)
## 第二步：添加工具（函数）

Now let's give our agent some useful capabilities. We will give it a weather function and a calculator. Both of these are functions we define in our code and provide them to the kernel as tools. 
现在让我们给我们的代理一些有用的能力。我们将给它一个天气函数和一个计算器。这两个都是我们在代码中定义的函数，并将它们作为工具提供给 Kernel。

In [ ]:
# Define useful functions
@kernel_function(description="Get current weather for a city")
def get_weather(city: str) -> str:
    """Mock weather function - replace with real API call."""
    weather_data = {
        "london": "Cloudy, 15°C",
        "paris": "Sunny, 22°C", 
        "tokyo": "Rainy, 18°C",
        "new york": "Partly cloudy, 20°C"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

@kernel_function(description="Calculate simple math expressions")
def calculate(expression: str) -> str:
    """Safe calculator for basic math."""
    try:
        # Only allow basic math operations for safety
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            result = eval(expression)
            return f"{expression} = {result}"
        else:
            return "Sorry, I can only do basic math operations (+, -, *, /, parentheses)"
    except:
        return "Sorry, I couldn't calculate that. Please check your expression."

# Add functions to kernel
kernel.add_function(plugin_name="tools", function=get_weather)
kernel.add_function(plugin_name="tools", function=calculate)

# Create enhanced agent
enhanced_agent = ChatCompletionAgent(
    kernel=kernel,
    name="SmartAssistant",
    instructions="""
    You are a helpful assistant with weather and calculator capabilities.
    
    - Use get_weather when asked about weather in specific cities
    - Use calculate for math problems
    - Be friendly and explain what you're doing
    """
)

print("✅ Enhanced agent created with tools!")

## Step 3: Test the Agent with Tools
## 第三步：使用工具测试代理

Let's see our agent use its tools:
让我们看看我们的代理如何使用它的工具：

In [ ]:
# Test weather function
print("🌤️ Testing weather function:")
response = await enhanced_agent.get_response("What's the weather in London?")
print(f"🤖 {enhanced_agent.name}: {response.content}\n")

# Test calculator function  
print("🧮 Testing calculator function:")
response = await enhanced_agent.get_response("What's 25 * 4 + 10?")
print(f"🤖 {enhanced_agent.name}: {response.content}\n")

# Test general conversation
print("💬 Testing general conversation:")
response = await enhanced_agent.get_response("Tell me a fun fact about AI")
print(f"🤖 {enhanced_agent.name}: {response.content}")

## Step 4: Conversation with Memory
## 第四步：带记忆的对话

For conversations that remember previous messages. We will use semantic kernel out of the box memory management, but that will depend on context window.
对于记住以前消息的对话。我们将使用 Semantic Kernel 开箱即用的内存管理，但这将取决于上下文窗口。

Memory can be solved by summarizing past conversations or use a longer term memory like RAG. There is also learning memory where we want the agent to elarn from all past interactions, this is also implemented using RAG where we store successful resolution examples and retrieve them for similar cases. 
内存可以通过总结过去的对话或使用像 RAG 这样的长期记忆来解决。还有一种学习记忆，我们希望代理从所有过去的交互中学习，这也是使用 RAG 实现的，我们存储成功解决的示例并在类似情况下检索它们。

In [ ]:
async def chat_with_memory():
    """Demonstrate conversation with memory."""
    
    print("💭 Conversation with Memory Demo")
    print("=" * 40)
    
    # Messages that build on each other
    messages = [
        "Hi! I'm planning a trip to Paris.",
        "What's the weather like there?",
        "That sounds nice! Can you calculate 150 * 7 for my budget?",
        "Perfect, that should cover my week there. Thanks!"
    ]
    
    thread = None  # This will store conversation history
    
    for msg in messages:
        print(f"👤 User: {msg}")
        
        # Agent remembers previous messages through the thread
        response = await enhanced_agent.get_response(messages=msg, thread=thread)
        print(f"🤖 {enhanced_agent.name}: {response.content}\n")
        
        # Update thread to keep conversation history
        thread = response.thread

# Run the conversation demo
await chat_with_memory()

## Step 5: See Tools in Action (Advanced)
## 第五步：查看工具的实际运行（高级）

Watch exactly what happens when your agent uses tools:
确切地观察当你的代理使用工具时会发生什么：

In [ ]:
async def show_tool_usage():
    """Show detailed tool execution."""
    
    print("🔧 Tool Usage Demo")
    print("=" * 25)
    
    # Callback to see tool calls
    async def log_tool_calls(message):
        from semantic_kernel.contents import FunctionCallContent, FunctionResultContent
        
        for item in message.items or []:
            if isinstance(item, FunctionCallContent):
                print(f"  🔧 Calling: {item.name}({item.arguments})")
            elif isinstance(item, FunctionResultContent):
                print(f"  ✅ Result: {item.result}")
    
    user_input = "What's the weather in Tokyo and what's 15 + 27?"
    print(f"👤 User: {user_input}\n")
    
    # Use invoke to see intermediate steps
    async for response in enhanced_agent.invoke(
        messages=user_input,
        on_intermediate_message=log_tool_calls
    ):
        print(f"\n🤖 Final Response: {response.content}")

# Run the tool usage demo
await show_tool_usage()

## Step 6: Streaming Responses
## 第六步：流式响应

For real-time responses (like ChatGPT):
对于实时响应（如 ChatGPT）：

In [ ]:
async def streaming_demo():
    """Show streaming responses."""
    
    print("\n🌊 Streaming Demo")
    print("=" * 20)
    
    print("👤 User: Write a short poem about coding")
    print("🤖 Assistant: ", end="", flush=True)
    
    # Stream response word by word
    async for chunk in enhanced_agent.invoke_stream(
        messages="Write a short poem about coding"
    ):
        print(chunk.content, end="", flush=True)
    
    print("\n")  # New line when done

# Run streaming demo
await streaming_demo()


## Summary: What You've Learned
## 总结：你学到了什么

In [ ]:
print("🎓 What You've Built:")
print("=" * 30)

summary = [
    "✅ Basic AI agent with OpenAI",
    "✅ Custom tools/functions for weather and math", 
    "✅ Conversation memory management",
    "✅ Tool execution monitoring",
    "✅ Real-time streaming responses"
]

for item in summary:
    print(item)

print("\n🚀 Key Concepts:")
concepts = {
    "Agent": "AI that can reason, remember, and use tools",
    "Kernel": "Manages AI services and functions",
    "Functions": "Tools that extend agent capabilities", 
    "Thread": "Maintains conversation history",
    "Streaming": "Real-time response generation"
}

for concept, description in concepts.items():
    print(f"• {concept}: {description}")

print("\n💡 Next Steps:")
print("• Try different models (gpt-4, gpt-3.5-turbo)")
print("• Create custom functions for your use case")
print("• Explore multi-agent conversations")
print("• Add guardrails for production safety")

## Step 7: Sequential Agent Orchestration
## 第七步：顺序代理编排

Now let's see how multiple agents can work together in a pipeline - each agent processes the output from the previous one. Notice how easy Semantic Kernel makes this for us:
现在让我们看看多个代理如何在管道中协同工作——每个代理处理前一个代理的输出。请注意 Semantic Kernel 使这对我们来说多么容易：

In [ ]:
# Import orchestration components
from semantic_kernel.agents import SequentialOrchestration
from semantic_kernel.agents.runtime import InProcessRuntime
from semantic_kernel.contents import ChatMessageContent

print("🔗 Setting up Sequential Agent Pipeline")
print("=" * 45)

In [ ]:
# Create specialized agents for a marketing pipeline
def create_marketing_pipeline():
    """Create three agents that work together sequentially."""
    
    # Agent 1: Extract key information
    concept_extractor = ChatCompletionAgent(
        name="ConceptExtractor",
        instructions="""
        You are a marketing analyst. Given a product description, identify:
        - Key features (bullet points)
        - Target audience 
        - Unique selling points
        
        Format your output clearly with headers.
        """,
        kernel=kernel
    )
    
    # Agent 2: Write marketing copy
    copywriter = ChatCompletionAgent(
        name="Copywriter", 
        instructions="""
        You are a marketing copywriter. Take the analysis provided and write 
        compelling marketing copy (around 100-150 words). Make it engaging 
        and highlight the key benefits. Output just the marketing copy.
        """,
        kernel=kernel
    )
    
    # Agent 3: Polish and format
    editor = ChatCompletionAgent(
        name="Editor",
        instructions="""
        You are an editor. Take the marketing copy and polish it:
        - Fix grammar and clarity
        - Ensure consistent tone
        - Make it more compelling
        - Output the final polished version
        """,
        kernel=kernel
    )
    
    return [concept_extractor, copywriter, editor]

# Create the pipeline
marketing_agents = create_marketing_pipeline()
print(f"✅ Created {len(marketing_agents)} specialized agents:")
for agent in marketing_agents:
    print(f"   • {agent.name}")

In [ ]:
# Set up callback to see each agent's work
def agent_callback(message: ChatMessageContent) -> None:
    """Show what each agent produces."""
    print(f"\n🤖 {message.name}:")
    print("-" * 30)
    print(message.content)
    print()

# Create the sequential orchestration
sequential_pipeline = SequentialOrchestration(
    members=marketing_agents,
    agent_response_callback=agent_callback
)

print("🔗 Sequential pipeline created!")

In [ ]:
# Run the sequential pipeline
async def run_marketing_pipeline():
    """Execute the sequential agent pipeline."""
    
    print("🚀 Running Marketing Pipeline")
    print("=" * 35)
    
    # Start the runtime
    runtime = InProcessRuntime()
    runtime.start()
    
    try:
        # Input: Product description
        product_description = (
            "A smart water bottle with temperature display, "
            "app connectivity, hydration reminders, and "
            "leak-proof design. Made from BPA-free materials."
        )
        
        print(f"📝 Input Product: {product_description}\n")
        print("Processing through pipeline...")
        
        # Run the sequential orchestration
        result = await sequential_pipeline.invoke(
            task=product_description,
            runtime=runtime
        )
        
        # Get final result
        final_output = await result.get(timeout=30)
        
        print("🎯 FINAL MARKETING COPY:")
        print("=" * 40)
        print(final_output)
        
    finally:
        # Clean up
        await runtime.stop_when_idle()

# Execute the pipeline
await run_marketing_pipeline()

In [ ]:
# Quick demo with a different product
async def quick_pipeline_demo():
    """Quick demo with another product."""
    
    print("\n🔄 Quick Pipeline Demo #2")
    print("=" * 30)
    
    runtime = InProcessRuntime()
    runtime.start()
    
    try:
        # Different product
        product = "Wireless earbuds with 30-hour battery, noise cancellation, and workout-proof design"
        
        print(f"📝 Product: {product}")
        
        # Simple pipeline without detailed logging
        simple_pipeline = SequentialOrchestration(members=marketing_agents)
        
        result = await simple_pipeline.invoke(
            task=product,
            runtime=runtime
        )
        
        final_copy = await result.get(timeout=30)
        print(f"\n🎯 Final Copy:\n{final_copy}")
        
    finally:
        await runtime.stop_when_idle()

# Run quick demo
await quick_pipeline_demo()


## Summary: What You've Learned
## 总结：你学到了什么

In [ ]:
print("🎓 Complete Tutorial Summary:")
print("=" * 35)

summary = [
    "✅ Basic AI agent with OpenAI",
    "✅ Custom tools/functions for weather and math", 
    "✅ Conversation memory management",
    "✅ Tool execution monitoring",
    "✅ Real-time streaming responses",
    "✅ Sequential agent orchestration"
]

for item in summary:
    print(item)

print("\n🚀 Key Concepts:")
concepts = {
    "Agent": "AI that can reason, remember, and use tools",
    "Kernel": "Manages AI services and functions",
    "Functions": "Tools that extend agent capabilities", 
    "Thread": "Maintains conversation history",
    "Streaming": "Real-time response generation",
    "Sequential Orchestration": "Pipeline where agents process output sequentially"
}

for concept, description in concepts.items():
    print(f"• {concept}: {description}")

print("\n💡 Agent Patterns:")
patterns = [
    "Single Agent: One agent handles entire workflow",
    "Sequential: Agents work in pipeline (A → B → C)",
    "Concurrent: Multiple agents work simultaneously", 
    "Manager: Central agent coordinates specialists",
    "Handoff: Agents pass control to each other"
]

for pattern in patterns:
    print(f"• {pattern}")

print("\n🎯 When to Use Sequential Orchestration:")
use_cases = [
    "• Document processing (extract → summarize → format)",
    "• Content creation (research → write → edit)", 
    "• Data analysis (collect → analyze → visualize)",
    "• Code review (analyze → suggest → validate)"
]

for use_case in use_cases:
    print(use_case)

---

**🔑 Key Takeaways:**
**🔑 关键要点：**
- **Single Agents**: Great for simple tasks and learning
- **单一代理**：非常适合简单任务和学习
- **Sequential Orchestration**: Perfect for multi-step workflows where each step builds on the previous
- **顺序编排**：非常适合每一步都建立在前一步基础上的多步工作流
- **Specialization**: Each agent focuses on what it does best
- **专业化**：每个代理都专注于其最擅长的领域
- **Pipeline Benefits**: Better quality through specialized processing
- **管道优势**：通过专业化处理获得更好的质量
- **Real-world Applications**: Document processing, content creation, data analysis
- **现实世界应用**：文档处理、内容创建、数据分析

This tutorial covers everything from basic agents to sophisticated multi-agent pipelines!
本教程涵盖了从基本代理到复杂多代理管道的所有内容！